In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
class Lda:
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        samples, features = X.shape
        self.classes = np.unique(y)
        self.means = {}
        self.prior = {}
        for c in self.classes:
            Xc = X[y == c]
            self.means[c] = Xc.mean(axis=0)
            self.prior[c] = Xc.shape[0] / samples
        cov = np.zeros((features, features))
        for c in self.classes:
            Xc = X[y == c]
            centered = Xc - self.means[c]
            cov += centered.T @ centered
        cov /= (samples - len(self.classes))
        self.cov = cov
        self.cov_inv = np.linalg.pinv(cov)
        return self
    
    def predict(self, X):
        X = np.asarray(X)
        preds = []
        for x in X:
            scores = {}
            for c in self.classes:
                mu = self.means[c]
                scores[c] = x @ self.cov_inv @ mu - 0.5 * mu @ self.cov_inv @ mu + np.log(self.prior[c])
            preds.append(max(scores, key=scores.get)) 
        return np.array(preds)


class Qda:
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.classes = np.unique(y)
        samples, features = X.shape
        self.means = {}
        self.prior = {}
        self.cov_inv = {}
        self.log_det = {}
        for c in self.classes:
            Xc = X[y == c]
            self.means[c] = Xc.mean(axis=0)
            self.prior[c] = Xc.shape[0] / samples
            centered = Xc - self.means[c]
            cov_c = (centered.T @ centered) / (Xc.shape[0] - 1) + np.eye(features) * 1e-4
            self.cov_inv[c] = np.linalg.pinv(cov_c)
            _, logdet = np.linalg.slogdet(cov_c)
            self.log_det[c] = logdet
        return self

    def predict(self, X):
        X = np.asarray(X)
        preds = []
        for x in X:
            scores = {}
            for c in self.classes:
                mu = self.means[c]
                diff = x - mu
                # Quadratic score: -0.5*(x - mu)^T * Sigma_c^(-1) * (x - mu) - 0.5*ln|Sigma_c| + ln(pi_c)
                scores[c] = -0.5 * (diff @ self.cov_inv[c] @ diff) - 0.5 * self.log_det[c] + np.log(self.prior[c])
            preds.append(max(scores, key=scores.get))
        return np.array(preds)


class LdaReducer:
    def __init__(self, n_components=2):
        self.n_components = n_components
        self.W = None

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        features = X.shape[1]
        classes = np.unique(y)
        overall_mean = X.mean(axis=0)

        S_W = np.zeros((features, features))
        S_B = np.zeros((features, features))

        for c in classes:
            Xc = X[y == c]
            mean_c = Xc.mean(axis=0)
            centered = Xc - mean_c
            S_W += centered.T @ centered
            
            n_c = Xc.shape[0]
            mean_diff = (mean_c - overall_mean).reshape(-1, 1)
            S_B += n_c * (mean_diff @ mean_diff.T)

        S_W += np.eye(features) * 1e-4  # numerical stabilizer
        eigvals, eigvecs = np.linalg.eig(np.linalg.pinv(S_W) @ S_B)
        idx = np.argsort(np.real(eigvals))[::-1]
        self.W = np.real(eigvecs[:, idx[:self.n_components]])
        return self

    def transform(self, X):
        return np.asarray(X) @ self.W


# ==============================================================================
# TASK 1: CLINICAL CORONARY RISK REGISTRY (SAheart.csv)
# ==============================================================================
df_heart = pd.read_csv("SAheart.csv")
for col in ['row.names', 'Unnamed: 0']:
    if col in df_heart.columns:
        df_heart.drop(columns=[col], inplace=True)

df_heart['famhist'] = df_heart['famhist'].map({'Present': 1, 'Absent': 0}).astype(int)

y_heart = df_heart['chd'].values
X_heart = df_heart.drop(columns=['chd']).values

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_heart, y_heart, test_size=0.30, random_state=42, stratify=y_heart
)

lda_clf = Lda().fit(X_train_h, y_train_h)
y_pred_lda = lda_clf.predict(X_test_h)

qda_clf = Qda().fit(X_train_h, y_train_h)
y_pred_qda = qda_clf.predict(X_test_h)

# Step 3: Comparative Performance Audit
print("\n--- LDA Classification Report ---")
print(classification_report(y_test_h, y_pred_lda, target_names=["No CHD (0)", "CHD (1)"]))

print("--- QDA Classification Report ---")
print(classification_report(y_test_h, y_pred_qda, target_names=["No CHD (0)", "CHD (1)"]))

audit_df = pd.DataFrame([
    {
        "Algorithm": "Scratch LDA",
        "Accuracy": f"{accuracy_score(y_test_h, y_pred_lda):.4f}",
        "Precision (CHD=1)": f"{precision_score(y_test_h, y_pred_lda):.4f}",
        "Recall (CHD=1)": f"{recall_score(y_test_h, y_pred_lda):.4f}",
        "F1-Score (CHD=1)": f"{f1_score(y_test_h, y_pred_lda):.4f}"
    },
    {
        "Algorithm": "Scratch QDA",
        "Accuracy": f"{accuracy_score(y_test_h, y_pred_qda):.4f}",
        "Precision (CHD=1)": f"{precision_score(y_test_h, y_pred_qda):.4f}",
        "Recall (CHD=1)": f"{recall_score(y_test_h, y_pred_qda):.4f}",
        "F1-Score (CHD=1)": f"{f1_score(y_test_h, y_pred_qda):.4f}"
    }
])
print("\n--- Summary Performance Table ---")
print(audit_df.to_string(index=False))

# Confusion Matrix Heatmaps
cm_lda = confusion_matrix(y_test_h, y_pred_lda)
cm_qda = confusion_matrix(y_test_h, y_pred_qda)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
sns.heatmap(cm_lda, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
            xticklabels=["Pred: 0", "Pred: 1"], yticklabels=["Actual: 0", "Actual: 1"])
axes[0].set_title(f"Custom LDA Confusion Matrix\nFalse Negatives: {cm_lda[1, 0]}", fontweight='bold')
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual Label")

sns.heatmap(cm_qda, annot=True, fmt="d", cmap="Oranges", cbar=False, ax=axes[1],
            xticklabels=["Pred: 0", "Pred: 1"])
axes[1].set_title(f"Custom QDA Confusion Matrix\nFalse Negatives: {cm_qda[1, 0]}", fontweight='bold')
axes[1].set_xlabel("Predicted")

plt.tight_layout()
plt.show()


# ==============================================================================
# TASK 2: MULTI-CLASS WINE PROFILE VIA SCRATCH LDA REDUCTION (Wine.csv)
# ==============================================================================
print("\n" + "=" * 75)
print("TASK 2: MULTI-CLASS WINE LDA REDUCTION & CLASSIFICATION")
print("=" * 75)

# Step 1: Data Engineering & Normalization
df_wine = pd.read_csv("Wine.csv")
possible_targets = ['Customer_Segment', 'Wine', 'Class', 'class', 'cultivar']
target_col = next((c for c in possible_targets if c in df_wine.columns), df_wine.columns[0])

X_wine = df_wine.drop(columns=[target_col]).values
y_wine = df_wine[target_col].values

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.20, random_state=42, stratify=y_wine
)

# Feature standardization
mean_val = X_train_w.mean(axis=0)
std_val = X_train_w.std(axis=0) + 1e-8
X_train_w_scaled = (X_train_w - mean_val) / std_val
X_test_w_scaled = (X_test_w - mean_val) / std_val

# Step 2: Scratch LDA Reduction to 2 Components
lda_reducer = LdaReducer(n_components=2)
lda_reducer.fit(X_train_w_scaled, y_train_w)
X_train_lda = lda_reducer.transform(X_train_w_scaled)
X_test_lda = lda_reducer.transform(X_test_w_scaled)

# Step 3: Logistic Regression Classifier on 2D Projections
clf = LogisticRegression(random_state=42)
clf.fit(X_train_lda, y_train_w)
y_pred_w = clf.predict(X_test_lda)

print("\n--- Multi-Class Classification Report (2D LDA Space) ---")
print(classification_report(y_test_w, y_pred_w))
print("Confusion Matrix:")
print(confusion_matrix(y_test_w, y_pred_w))

# Step 4: 2D Cluster Visualization
viz_df = pd.DataFrame({
    'LD1': X_test_lda[:, 0],
    'LD2': X_test_lda[:, 1],
    'Cultivar': y_test_w
})

plt.figure(figsize=(8, 5.5))
sns.scatterplot(
    data=viz_df,
    x='LD1',
    y='LD2',
    hue='Cultivar',
    style='Cultivar',
    palette="Set1",
    s=110,
    edgecolor='k',
    alpha=0.9
)
plt.title("2D Projection of Wine Cultivars via Scratch LDA (Test Set)", fontweight='bold')
plt.xlabel("Linear Discriminant 1 (LD1)")
plt.ylabel("Linear Discriminant 2 (LD2)")
plt.legend(title="Cultivar", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: Sample documents
documents = [
    "Machine learning is a part of artificial intelligence",
    "Deep learning uses neural networks",
    "Machine learning uses algorithms and data",
    "Artificial intelligence is used in robotics",
    "Neural networks are used in deep learning"
]

# Step 2: Convert documents into TF-IDF matrix
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(documents)

print("TF-IDF Matrix Shape:", X.shape)

# Step 3: Apply SVD
lsa = TruncatedSVD(n_components=2, random_state=42)
X_lsa = lsa.fit_transform(X)

print("\nDocument Vectors:")
print(X_lsa)

# Step 4: Display topics using important words
terms = vectorizer.get_feature_names_out()

for i, component in enumerate(lsa.components_):
    words = component.argsort()[-5:][::-1]
    print(f"\nTopic {i + 1}:")
    print([terms[j] for j in words])

# Step 5: Calculate document similarity
similarity = cosine_similarity(X_lsa)

print("\nDocument Similarity:")
print(np.round(similarity, 2))

# Step 6: Find most similar document to document 1
scores = similarity[0]
scores[0] = -1  # Exclude document 1 itself

closest = np.argmax(scores)

print("\nDocument 1:")
print(documents[0])

print("\nMost Similar Document:")
print(documents[closest])